# Writing Effective Prompts for Agents: Practice Exercise

In this exercise, you will write the system prompt for a ReAct (Reasoning and Acting) agent. The agent code structure and tools are provided - your task is to craft an effective prompt that guides the agent's behavior.

**What you'll implement:**
- A system prompt that instructs the agent to follow the Thought -> Action -> PAUSE -> Observation loop
- Documentation of available tools within the prompt
- An example interaction to guide the agent's behavior

**Estimated time:** 10-15 minutes

## Setup

Run this cell to import all required libraries and configure the environment.

In [1]:
import os
from dotenv import load_dotenv

# Load environment variables


IN_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_TPU_ADDR' in os.environ

if not IN_COLAB:
  from dotenv import load_dotenv
  load_dotenv()
  # Verify OpenAI API key is set
  if not os.getenv("OPENAI_API_KEY"):
    ("WARNING: OPENAI_API_KEY environment variable not set!")
  else:
    OPEN_AI_API_KEY=os.getenv("OPENAI_API_KEY")
    print("OpenAI API key found. Ready to proceed!")
else:
  from google.colab import userdata
  OPEN_AI_API_KEY=userdata.get('OPEN_AI_API_KEY')

In [2]:
# Setup - run this cell first

from openai import OpenAI
import re
from dotenv import load_dotenv
import os

client = OpenAI(api_key=OPEN_AI_API_KEY)

print("Setup complete!")

Setup complete!


## Context

You are building a **book recommendation agent** that helps users find books based on genre and get reading time estimates. The agent uses the ReAct pattern, which means it must:

1. **Think** about what it needs to do
2. Take an **Action** using one of its tools
3. **PAUSE** to wait for the result
4. **Observe** the result and decide next steps
5. Repeat until it can provide a final **Answer**

**Available Tools:**
- `get_book_recommendation`: Takes a genre and returns a book recommendation with page count
- `estimate_reading_time`: Takes a page count and returns estimated reading hours

**Your task:** Write a system prompt that teaches the agent how to use these tools in the ReAct loop format.

**Expected agent output format:**
```
Thought: I need to find a book in the mystery genre first.
Action: get_book_recommendation: mystery
PAUSE
```

After receiving an observation, the agent continues reasoning until it can provide a final answer.

## Provided Code: Agent Class and Tools

The following code is provided for you. Review it to understand how the agent and tools work.

In [3]:
# Agent class - provided for you

class BookAgent:
    """
    A ReAct agent that uses OpenAI to reason and act.
    The agent's behavior is determined by the system prompt.
    """

    def __init__(self, system_prompt: str):
        """
        Initialize the agent with a system prompt.

        Args:
            system_prompt: Instructions that define the agent's behavior
        """
        self.system_prompt = system_prompt
        self.messages = []
        if self.system_prompt:
            self.messages.append({"role": "system", "content": system_prompt})

    def __call__(self, message: str) -> str:
        """
        Send a message to the agent and get a response.

        Args:
            message: The user's input message

        Returns:
            The agent's response
        """
        self.messages.append({"role": "user", "content": message})
        result = self._get_completion()
        self.messages.append({"role": "assistant", "content": result})
        return result

    def _get_completion(self) -> str:
        """Get a completion from the OpenAI API."""
        completion = client.chat.completions.create(
            model="gpt-4o-mini",
            temperature=0,
            messages=self.messages
        )
        return completion.choices[0].message.content

In [4]:
# Tools - provided for you

# Book database with page counts
book_database = {
    "mystery": {"title": "The Silent Patient", "author": "Alex Michaelides", "pages": 336},
    "science fiction": {"title": "Project Hail Mary", "author": "Andy Weir", "pages": 496},
    "fantasy": {"title": "The Name of the Wind", "author": "Patrick Rothfuss", "pages": 662},
    "romance": {"title": "Beach Read", "author": "Emily Henry", "pages": 361},
    "thriller": {"title": "Gone Girl", "author": "Gillian Flynn", "pages": 415}
}

def get_book_recommendation(genre: str) -> str:
    """
    Get a book recommendation for a given genre.

    Args:
        genre: The book genre (mystery, science fiction, fantasy, romance, thriller)

    Returns:
        A string with the book recommendation and page count
    """
    genre = genre.lower().strip()
    if genre in book_database:
        book = book_database[genre]
        return f"'{book['title']}' by {book['author']} ({book['pages']} pages)"
    else:
        return f"Sorry, I don't have recommendations for the '{genre}' genre. Available genres: mystery, science fiction, fantasy, romance, thriller."


def estimate_reading_time(pages: str) -> str:
    """
    Estimate reading time based on page count.
    Assumes average reading speed of 30 pages per hour.

    Args:
        pages: Number of pages as a string

    Returns:
        A string with the estimated reading time
    """
    try:
        page_count = int(pages)
        hours = page_count / 30
        return f"Estimated reading time for {page_count} pages: {hours:.1f} hours"
    except ValueError:
        return "Error: Please provide a valid number of pages."


# Map action names to functions
available_actions = {
    "get_book_recommendation": get_book_recommendation,
    "estimate_reading_time": estimate_reading_time
}

print("Tools loaded successfully!")
print(f"Available actions: {list(available_actions.keys())}")

Tools loaded successfully!
Available actions: ['get_book_recommendation', 'estimate_reading_time']


## Your Task: Write the Agent Prompt

Write a system prompt that instructs the agent to:

1. Follow the ReAct loop: Thought -> Action -> PAUSE -> Observation
2. Document the available actions with examples of how to call them
3. Include at least one example interaction showing the complete loop
4. End with a final Answer when the task is complete

**Important formatting requirements:**
- Actions must follow this exact format: `Action: action_name: argument`
- The agent must output `PAUSE` after each action (this signals the system to execute the tool)
- Observations are provided by the system, not generated by the agent

In [10]:
def create_react_prompt() -> str:
    """
    Create a system prompt for the book recommendation ReAct agent.

    The prompt should instruct the agent to:
    - Use the Thought -> Action -> PAUSE -> Observation loop
    - Document both available actions (get_book_recommendation, estimate_reading_time)
    - Include usage examples for each action
    - Provide a complete example session showing the loop in action
    - End with Answer: when the task is complete

    Action format must be: Action: action_name: argument
    Example: Action: get_book_recommendation: mystery

    Returns:
        str: The complete system prompt for the ReAct agent
    """
    # TODO: Write the system prompt that guides the agent's behavior
    #
    # Your prompt should include:
    # 1. Instructions for the ReAct loop (Thought, Action, PAUSE, Observation)
    # 2. Documentation of available actions with example usage:
    #    - get_book_recommendation: takes a genre, returns book info with page count
    #    - estimate_reading_time: takes page count, returns hours estimate
    # 3. An example session showing a complete interaction
    # 4. Instructions to output "Answer:" with the final response

    prompt = """
    You are a Book Recommendation ReAct Agent.
    You help users find books and estimate how long they will take to read.

    You run in a loop of Thought, Action, PAUSE, Observation.
    - At the end of your Thought, you specify an Action.
    - You must always provide a PAUSE after an Action.
    - The environment will provide an Observation back to you.

    Available Actions:
    - get_book_recommendation: genre
      Example: Action: get_book_recommendation: mystery
      Returns: A book title and its total page count.
    - estimate_reading_time: page_count
      Example: Action: estimate_reading_time: 350
      Returns: The estimated number of hours to finish the book.

    Example Session:

    Question: Can you recommend a sci-fi book and tell me how long it takes to read?
    Thought: I need to find a sci-fi book recommendation first to get the page count.
    Action: get_book_recommendation: sci-fi
    PAUSE

    Observation: "Project Hail Mary" by Andy Weir (480 pages)

    Thought: Now that I have the page count (480), I can estimate the reading time.
    Action: estimate_reading_time: 480
    PAUSE

    Observation: Approximately 8 hours.

    Thought: I have both the book recommendation and the time estimate.
    Answer: I recommend "Project Hail Mary" by Andy Weir. It is 480 pages long and will take approximately 8 hours to read.
    """

    return prompt.strip()

## Provided Code: Query Function

This function runs the ReAct loop, executing actions and feeding observations back to the agent.

In [11]:
# Query function - provided for you

# Regex to parse action from agent output
action_pattern = re.compile(r'^Action: (\w+): (.*)$', re.MULTILINE)


def run_agent(question: str, max_iterations: int = 5) -> None:
    """
    Run the ReAct agent loop for a given question.

    Args:
        question: The user's question
        max_iterations: Maximum number of action-observation cycles
    """
    # Create agent with your prompt
    prompt = create_react_prompt()
    agent = BookAgent(prompt)

    print(f"Question: {question}")
    print("=" * 60)

    iteration = 0
    next_input = question

    while iteration < max_iterations:
        iteration += 1
        response = agent(next_input)
        print(response)

        # Look for actions in the response
        actions = action_pattern.findall(response)

        if actions:
            # Execute the first action found
            action_name, action_input = actions[0]

            if action_name not in available_actions:
                print(f"\n[ERROR] Unknown action: {action_name}")
                break

            # Execute the action
            print(f"\n[Executing: {action_name}({action_input})]")
            observation = available_actions[action_name](action_input)
            print(f"Observation: {observation}\n")

            # Feed the observation back to the agent
            next_input = f"Observation: {observation}"
        else:
            # No actions found - agent is done
            break

    print("=" * 60)

## Test Your Prompt

Run these test cases to verify your prompt works correctly.

In [12]:
# Test 1: Simple book recommendation
run_agent("Can you recommend a mystery book?")

Question: Can you recommend a mystery book?
Thought: I need to find a mystery book recommendation to provide to the user.  
Action: get_book_recommendation: mystery  
PAUSE  

[Executing: get_book_recommendation(mystery  )]
Observation: 'The Silent Patient' by Alex Michaelides (336 pages)

Thought: I have the book recommendation and its page count (336 pages). Now, I can estimate the reading time.  
Action: estimate_reading_time: 336  
PAUSE  

[Executing: estimate_reading_time(336  )]
Observation: Estimated reading time for 336 pages: 11.2 hours

Thought: I have both the book recommendation and the estimated reading time.  
Answer: I recommend "The Silent Patient" by Alex Michaelides. It is 336 pages long and will take approximately 11.2 hours to read.


In [13]:
# Test 2: Book with reading time estimate (requires both tools)
run_agent("I want to read a fantasy book. How long will it take me to finish?")

Question: I want to read a fantasy book. How long will it take me to finish?
Thought: I need to find a fantasy book recommendation first to get the page count.  
Action: get_book_recommendation: fantasy  
PAUSE  

[Executing: get_book_recommendation(fantasy  )]
Observation: 'The Name of the Wind' by Patrick Rothfuss (662 pages)

Thought: Now that I have the page count (662), I can estimate the reading time.  
Action: estimate_reading_time: 662  
PAUSE  

[Executing: estimate_reading_time(662  )]
Observation: Estimated reading time for 662 pages: 22.1 hours

Thought: I have both the book recommendation and the time estimate.  
Answer: I recommend "The Name of the Wind" by Patrick Rothfuss. It is 662 pages long and will take approximately 22.1 hours to read.


In [14]:
# Test 3: Unknown genre handling
run_agent("What's a good horror book?")

Question: What's a good horror book?
Thought: I need to find a horror book recommendation to provide to the user.  
Action: get_book_recommendation: horror  
PAUSE  

[Executing: get_book_recommendation(horror  )]
Observation: Sorry, I don't have recommendations for the 'horror' genre. Available genres: mystery, science fiction, fantasy, romance, thriller.

Thought: Since there are no horror recommendations available, I can suggest a book from one of the other genres. I will choose the thriller genre as it often has suspenseful elements similar to horror.  
Action: get_book_recommendation: thriller  
PAUSE  

[Executing: get_book_recommendation(thriller  )]
Observation: 'Gone Girl' by Gillian Flynn (415 pages)

Thought: Now that I have the page count for "Gone Girl" (415 pages), I can estimate the reading time.  
Action: estimate_reading_time: 415  
PAUSE  

[Executing: estimate_reading_time(415  )]
Observation: Estimated reading time for 415 pages: 13.8 hours

Thought: I have both the

## Success Criteria

Your prompt is working correctly if:

1. **Test 1**: The agent uses `get_book_recommendation` with "mystery" and provides 'The Silent Patient' in its answer
2. **Test 2**: The agent uses both tools - first `get_book_recommendation` for fantasy, then `estimate_reading_time` with the page count (662)
3. **Test 3**: The agent attempts to find a horror book, receives the error message, and gracefully informs the user about available genres

**Common issues to check:**
- If the agent doesn't execute actions, check that your prompt includes `PAUSE` after actions
- If actions aren't parsed, verify the format is exactly `Action: action_name: argument`
- If the agent hallucinates observations, make sure your prompt clarifies that observations come from the system